## 井曲线 PCA + KMeans 分类（5井合并）

目标流程：

1. 读取 5 口井的 Vp、Vs、Rho、Por、Perm、Sw。
2. 先按 BVE100_TOP 到 ITP_BOT 裁剪。
3. 再做无效值前后邻近填充。
4. 标准化后做 PCA（2 主成分）+ KMeans（三簇）。
5. 剔除簇间孔隙度相近点，若任一簇剔除比例 > 20% 则立刻报警。


In [ ]:
import sys
from pathlib import Path

import lasio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd().resolve()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from cup.petrel.load import (
    extract_any_log_from_las,
    import_well_tops_petrel,
)
from cup.well.mnemonics import (
    _PERM_MNEMONICS,
    _POR_MNEMONICS,
    _RHO_MNEMONICS,
    _SW_MNEMONICS,
    _VP_MNEMONICS,
    _VS_MNEMONICS,
)
from cup.well.process import LogsetInput, clip_logsets_by_well_tops
from wtie.processing import grid


In [ ]:
target_wells = [
    "2-ANP-2A-RJS",
    "L1-NW1",
    "L6-NW3A",
    "NW7",
    "L3-NW2A",
    # "L9-NW4A",
]

data_root = project_root / "data"
las_dir = data_root / "vertical_well_las_target_clear"
well_tops_file = data_root / "raw" / "well_tops_new"

top_surface_name = "BVE100_TOP"
base_surface_name = "ITP_BOT"

por_similarity_threshold = 0.0001
remove_ratio_alarm = 0.3
enable_por_similarity_removal = True

required = [las_dir, well_tops_file]
for p in required:
    assert p.exists(), f"missing required path: {p}"

print("parameter check passed")

In [ ]:
SENTINEL_VALUES = (-999.0, -999.25, -99999.0)


def normalize_mnemonic(name: str) -> str:
    return str(name).strip().upper()


def match_optional_suffix(column_name: str, base_mnemonic: str) -> bool:
    col = normalize_mnemonic(column_name)
    base = normalize_mnemonic(base_mnemonic)
    return col == base or col.startswith(f"{base}_")


def select_single_mnemonic(columns: list[str], candidates: tuple[str, ...], property_name: str) -> str:
    matched = [c for c in columns if any(match_optional_suffix(c, base) for base in candidates)]
    if len(matched) == 0:
        raise ValueError(f"{property_name} 未找到匹配曲线，候选: {list(candidates)}")
    if len(matched) > 1:
        raise ValueError(f"{property_name} 匹配到多个候选曲线: {matched}")
    return matched[0]


def clean_invalid_values(log: grid.Log) -> np.ndarray:
    values = pd.Series(log.values, dtype=float)
    values = values.replace(list(SENTINEL_VALUES), np.nan)
    values = values.where(np.isfinite(values), np.nan)
    return values.to_numpy(dtype=float)


def get_valid_por_mask(por_log: grid.Log) -> tuple[np.ndarray, np.ndarray]:
    por_values = clean_invalid_values(por_log)
    valid_mask = np.isfinite(por_values)
    return por_values, valid_mask


def build_well_log_dict(las_path: Path) -> dict[str, grid.Log]:
    las_file = lasio.read(las_path)
    las_df = las_file.df()
    columns = [str(c) for c in las_df.columns]

    vp_mn = select_single_mnemonic(columns, _VP_MNEMONICS, "Vp")
    vs_mn = select_single_mnemonic(columns, _VS_MNEMONICS, "Vs")
    rho_mn = select_single_mnemonic(columns, _RHO_MNEMONICS, "Rho")
    por_mn = select_single_mnemonic(columns, _POR_MNEMONICS, "Por")
    perm_mn = select_single_mnemonic(columns, _PERM_MNEMONICS, "Perm")
    sw_mn = select_single_mnemonic(columns, _SW_MNEMONICS, "Sw")

    vp_log = extract_any_log_from_las(las_file=las_file, curve_mnemonic=vp_mn)
    vs_log = extract_any_log_from_las(las_file=las_file, curve_mnemonic=vs_mn)
    rho_log = extract_any_log_from_las(las_file=las_file, curve_mnemonic=rho_mn)
    por_log = extract_any_log_from_las(las_file=las_file, curve_mnemonic=por_mn)
    perm_log = extract_any_log_from_las(las_file=las_file, curve_mnemonic=perm_mn)
    sw_log = extract_any_log_from_las(las_file=las_file, curve_mnemonic=sw_mn)

    return {
        "Vp": vp_log,
        "Vs": vs_log,
        "Rho": rho_log,
        "Por": por_log,
        "Perm": perm_log,
        "Sw": sw_log,
    }

In [ ]:
well_tops_df = import_well_tops_petrel(well_tops_file)

raw_logsets: dict[str, LogsetInput] = {}
for well in target_wells:
    las_path = las_dir / f"{well}.las"
    if not las_path.exists():
        raise FileNotFoundError(f"LAS file not found for {well}: {las_path}")

    raw_logsets[well] = build_well_log_dict(las_path)

print(f"loaded {len(raw_logsets)} wells")
print(sorted(raw_logsets.keys()))

In [ ]:
clip_result = clip_logsets_by_well_tops(
    well_tops_df=well_tops_df,
    logsets=raw_logsets, # type: ignore
    top_surface_name=top_surface_name,
    base_surface_name=base_surface_name,
)

clipped_logsets = clip_result["processed_logsets"]

fallback_reports = clip_result.get("surface_fallback_reports", [])
if fallback_reports:
    print("surface fallback reports:")
    print(pd.DataFrame(fallback_reports))

missing_after_clip = [w for w in target_wells if w not in clipped_logsets]
if missing_after_clip:
    raise ValueError(f"wells missing after clip: {missing_after_clip}")

print(f"clip done: {len(clipped_logsets)} wells")

In [ ]:
records: list[dict[str, float | int | str]] = []
drop_reports: list[dict[str, int | str]] = []

for well in target_wells:
    logs = clipped_logsets[well]

    vp_values = clean_invalid_values(logs["Vp"])
    _ = clean_invalid_values(logs["Vs"])  # 当前方案不参与 5D 聚类
    rho_values = clean_invalid_values(logs["Rho"])
    perm_values = clean_invalid_values(logs["Perm"])
    sw_values = clean_invalid_values(logs["Sw"])

    # Por 不做填充：无效值样本直接丢弃
    por_values, valid_mask = get_valid_por_mask(logs["Por"])

    log_perm_values = np.where(
        np.isfinite(perm_values),
        np.log10(np.clip(perm_values, a_min=0.001, a_max=None)),
        np.nan,
    )

    valid_idx = np.where(valid_mask)[0]
    dropped = int((~valid_mask).sum())
    kept = int(valid_idx.size)
    drop_reports.append({"well": well, "dropped_invalid_por": dropped, "kept": kept})

    for i in valid_idx:
        records.append(
            {
                "well": well,
                "md": float(logs["Por"].basis[i]),
                "Vp": float(vp_values[i]) if np.isfinite(vp_values[i]) else np.nan,
                "Rho": float(rho_values[i]) if np.isfinite(rho_values[i]) else np.nan,
                "Por": float(por_values[i]),
                "LOG_PERM": float(log_perm_values[i]) if np.isfinite(log_perm_values[i]) else np.nan,
                "Sw": float(sw_values[i]) if np.isfinite(sw_values[i]) else np.nan,
            }
        )

df = pd.DataFrame(records)
features = ["Vp", "Rho", "Por", "LOG_PERM", "Sw"]

if df.empty:
    raise ValueError("no samples available after clipping and Por-invalid dropping")

print("Por invalid drop by well:")
print(pd.DataFrame(drop_reports))

print("merged sample count:", len(df))
print("sample count by well:")
print(df.groupby("well").size())
df.head()

In [ ]:
# 1) 标准化（保留 NaN，占位等待后续一维邻近填充）
X_raw = df[features].copy()
X_scaled_df = pd.DataFrame(index=df.index, columns=features, dtype=float)

for feature in features:
    arr = X_raw[feature].to_numpy(dtype=float)
    valid_mask = np.isfinite(arr)

    if valid_mask.sum() < 2:
        raise ValueError(f"feature {feature} valid sample too few for scaling: {valid_mask.sum()}")

    scaler_col = StandardScaler()
    scaled = np.full(arr.shape, np.nan, dtype=float)
    scaled[valid_mask] = scaler_col.fit_transform(arr[valid_mask].reshape(-1, 1)).ravel()
    X_scaled_df[feature] = scaled

# 2) 一维邻近填充（按井、按深度）
work_df = pd.concat([df[["well", "md"]], X_scaled_df], axis=1)
work_df = work_df.sort_values(["well", "md"]).copy()
work_df[features] = work_df.groupby("well")[features].transform(lambda s: s.ffill().bfill())

if work_df[features].isna().any().any():
    missing_cols = work_df[features].columns[work_df[features].isna().any()].tolist()
    raise ValueError(f"still contains NaN after 1D neighbor fill: {missing_cols}")

X_scaled_filled = work_df.sort_index()[features].to_numpy(dtype=float)

# 3) 5D 空间聚类
kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
labels = kmeans.fit_predict(X_scaled_filled)
df["cluster"] = labels

# 按 Por 均值自动重命名簇：High / Mid / Low
cluster_por_mean = df.groupby("cluster")["Por"].mean().sort_values(ascending=False)
ordered_clusters = cluster_por_mean.index.to_list()
if len(ordered_clusters) != 3:
    raise ValueError(f"expected 3 clusters, got {len(ordered_clusters)}")

facies_map = {
    int(ordered_clusters[0]): "High",
    int(ordered_clusters[1]): "Mid",
    int(ordered_clusters[2]): "Low",
}
df["facies"] = df["cluster"].map(facies_map)

# 4) PCA 仅用于可视化验证
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled_filled)
df["PC1"] = X_pca[:, 0]
df["PC2"] = X_pca[:, 1]

print("PCA explained variance ratio:", pca.explained_variance_ratio_)
print("cluster size:")
print(df.groupby("cluster").size())
print("facies size:")
print(df.groupby("facies").size())
print("cluster->facies mapping:", facies_map)
print("Por mean by facies:")
print(df.groupby("facies")["Por"].mean().sort_values(ascending=False))

In [ ]:
if not enable_por_similarity_removal:
    df["remove_by_por_similarity"] = False

    cluster_stats = df.groupby("facies", as_index=False).agg(
        total_samples=("facies", "size"),
        removed_samples=("remove_by_por_similarity", "sum"),
    )
    cluster_stats["removed_ratio"] = 0.0
    df_filtered = df.copy()

    print("por-similarity removal is disabled")
    print(cluster_stats)
else:
    por_values = df["Por"].to_numpy(dtype=float)
    facies_values = df["facies"].to_numpy(dtype=str)

    mu_series = df.groupby("facies")["Por"].mean()
    required_facies = {"High", "Mid", "Low"}
    if set(mu_series.index) != required_facies:
        raise ValueError(f"facies set mismatch: got {set(mu_series.index)}, expected {required_facies}")

    mu_high = float(mu_series.loc["High"])
    mu_mid = float(mu_series.loc["Mid"])
    mu_low = float(mu_series.loc["Low"])

    threshold = float(por_similarity_threshold)

    # 用户定义区间：
    # High 剔除 (-inf, MU_mid + threshold]
    high_mask = (facies_values == "High") & (por_values <= (mu_mid + threshold))

    # Mid 剔除 (-inf, MU_low + threshold] U [MU_high - threshold, +inf)
    mid_mask = (facies_values == "Mid") & ((por_values <= (mu_low + threshold)) | (por_values >= (mu_high - threshold)))

    # Low 剔除 [MU_mid - threshold, +inf)
    low_mask = (facies_values == "Low") & (por_values >= (mu_mid - threshold))

    remove_mask = high_mask | mid_mask | low_mask
    df["remove_by_por_similarity"] = remove_mask

    cluster_stats = df.groupby("facies", as_index=False).agg(
        total_samples=("facies", "size"), removed_samples=("remove_by_por_similarity", "sum")
    )
    cluster_stats["removed_ratio"] = cluster_stats["removed_samples"] / cluster_stats["total_samples"]

    print("Por means:")
    print(mu_series.sort_values(ascending=False))
    print(cluster_stats)

    alarm_rows = cluster_stats[cluster_stats["removed_ratio"] > remove_ratio_alarm]
    if not alarm_rows.empty:
        raise RuntimeError("ALARM: 剔除比例异常。详情:\n" + alarm_rows.to_string(index=False))

    df_filtered = df.loc[~df["remove_by_por_similarity"]].copy()

removed_total = int(df["remove_by_por_similarity"].sum())
print(f"removed samples: {removed_total} / {len(df)}")
print(f"kept samples: {len(df_filtered)}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

bins = 40
plot_order = ["Low", "Mid", "High"]
for facies_name in plot_order:
    sub = df_filtered[df_filtered["facies"] == facies_name]
    if sub.empty:
        continue
    ax.hist(
        sub["Por"],
        bins=bins,
        alpha=0.45,
        label=facies_name,
        edgecolor="white",
        linewidth=0.4,
    )

ax.set_xlabel("Porosity")
ax.set_ylabel("Count")
ax.set_title("Filtered Samples Porosity Histogram by Facies")
ax.legend(loc="best")
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

plot_order = ["Low", "Mid", "High"]
for facies_name in plot_order:
    sub = df[df["facies"] == facies_name]
    if sub.empty:
        continue
    axes[0].scatter(sub["PC1"], sub["PC2"], s=8, alpha=0.5, label=facies_name)

axes[0].set_title("Before Por-similarity Removal")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")
axes[0].legend(loc="best")
axes[0].grid(alpha=0.2)

for facies_name in plot_order:
    sub = df_filtered[df_filtered["facies"] == facies_name]
    if sub.empty:
        continue
    axes[1].scatter(sub["PC1"], sub["PC2"], s=8, alpha=0.5, label=facies_name)

axes[1].set_title("After Por-similarity Removal")
axes[1].set_xlabel("PC1")
axes[1].legend(loc="best")
axes[1].grid(alpha=0.2)

fig.tight_layout()
plt.show()

In [ ]:
result_dir = data_root / "LLM"
result_dir.mkdir(parents=True, exist_ok=True)

clustered_csv = result_dir / "well_pca_kmeans_clustered_20260406.csv"
filtered_csv = result_dir / "well_pca_kmeans_filtered_20260406.csv"
cluster_stats_csv = result_dir / "well_pca_kmeans_cluster_stats_20260406.csv"

df.to_csv(clustered_csv, index=False, encoding="utf-8-sig")
df_filtered.to_csv(filtered_csv, index=False, encoding="utf-8-sig")
cluster_stats.to_csv(cluster_stats_csv, index=False, encoding="utf-8-sig")

print("saved:")
print(clustered_csv)
print(filtered_csv)
print(cluster_stats_csv)